01. Setup

In [1]:
# Instalação do Java e Spark
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark

# Variáveis de ambiente
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

# Pacote Iceberg
!wget -q https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.6.1/iceberg-spark-runtime-3.5_2.12-1.6.1.jar -P /content/spark-3.5.0-bin-hadoop3/jars/

import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col

# Sessão Spark
spark = SparkSession.builder \
    .appName("IcebergWithSpark") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.6.1,org.postgresql:postgresql:42.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/content/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

# Diretório do Warehouse
!mkdir -p /content/warehouse

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.8 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [

# Exemplo básico

In [16]:
# Criando Vendas
spark.sql("DROP TABLE IF EXISTS hadoop_catalog.default.vendas")

spark.sql("""
    CREATE TABLE hadoop_catalog.default.vendas (
        id INT,
        produto STRING,
        quantidade INT,
        preco DOUBLE,
        data_venda DATE
    )
    USING iceberg
""")

DataFrame[]

In [17]:
# Incluindo dados
data = [
    (1, "Produto A", 10, 15.5, "2024-11-01"),
    (2, "Produto B", 5, 22.0, "2024-11-02"),
    (3, "Produto C", 8, 30.0, "2024-11-03")
]
columns = ["id", "produto", "quantidade", "preco", "data_venda"]
df = spark.createDataFrame(data, columns)
df = df.withColumn("data_venda", to_date(col("data_venda"), "yyyy-MM-dd"))

df.writeTo("hadoop_catalog.default.vendas").append()

In [18]:
# Vizualizando Dados
spark.sql("SELECT * FROM hadoop_catalog.default.vendas").show()

+---+---------+----------+-----+----------+
| id|  produto|quantidade|preco|data_venda|
+---+---------+----------+-----+----------+
|  1|Produto A|        10| 15.5|2024-11-01|
|  2|Produto B|         5| 22.0|2024-11-02|
|  3|Produto C|         8| 30.0|2024-11-03|
+---+---------+----------+-----+----------+



### Uso de Metadados

In [19]:
# 1. Visualizando schemas do catálogo
print("Schemas no catálogo:")
spark.sql("""
    SHOW SCHEMAS IN hadoop_catalog
""").show()

Schemas no catálogo:
+---------+
|namespace|
+---------+
|  default|
+---------+



In [20]:
# 2. Visualizando tabelas do schema default
print("Tabelas no schema default:")
spark.sql("""
    SHOW TABLES IN hadoop_catalog.default
""").show()

Tabelas no schema default:
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default|   vendas|      false|
+---------+---------+-----------+



In [21]:
# 3. Schema da tabela vendas
print("Schema da tabela vendas:")
spark.sql("""
    DESCRIBE TABLE hadoop_catalog.default.vendas
""").show()

Schema da tabela vendas:
+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|        id|      int|   NULL|
|   produto|   string|   NULL|
|quantidade|      int|   NULL|
|     preco|   double|   NULL|
|data_venda|     date|   NULL|
+----------+---------+-------+



In [22]:
# 4. Propriedades da tabela 'vendas'
print("Propriedades da tabela vendas:")
spark.sql("""
    DESCRIBE EXTENDED hadoop_catalog.default.vendas
""").show(truncate=False)

Propriedades da tabela vendas:
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|id                          |int                                                                                                                   |NULL   |
|produto                     |string                                                                                                                |NULL   |
|quantidade                  |int                                                                                                                   |NULL   |
|preco               

In [23]:
# 5. Listando tabelas de metadados para vendas
metadata_tables = ["snapshots", "manifests", "partitions", "files", "history", "refs"]

for table in metadata_tables:
    print(f"\nMetadados da tabela '{table}':")
    spark.sql(f"""
        SELECT * FROM hadoop_catalog.default.vendas.{table}
    """).show(truncate=False)


Metadados da tabela 'snapshots':
+-----------------------+-------------------+---------+---------+---------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                  |summary                                                                                                                                                  

In [24]:
# 6. Catálogos disponíveis
print("Catálogos:")
spark.sql("""
    SHOW CATALOGS
""").show()


Catálogos:
+--------------------+
|             catalog|
+--------------------+
|default_cache_ice...|
|      hadoop_catalog|
|       spark_catalog|
+--------------------+



In [25]:
# 7. Visualizando catálogo hadoop_catalog
print("Visualizando hadoop_catalog:")
catalog_conf = spark.sparkContext.getConf().getAll()
for key, value in catalog_conf:
    if 'hadoop_catalog' in key:
        print(f"{key}: {value}")

Visualizando hadoop_catalog:
spark.sql.catalog.hadoop_catalog: org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.hadoop_catalog.warehouse: /content/warehouse
spark.sql.catalog.hadoop_catalog.type: hadoop


In [26]:
# 8. Novo schema sales
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS hadoop_catalog.sales
""")

DataFrame[]

In [27]:
# Listando os schemas
print("Schemas no hadoop_catalog após adicionar sales:")
spark.sql("""
    SHOW SCHEMAS IN hadoop_catalog
""").show()

Schemas no hadoop_catalog após adicionar sales:
+---------+
|namespace|
+---------+
|  default|
|    sales|
+---------+



In [28]:
# 9. Criando tabela no schema sales
spark.sql("""
    CREATE TABLE hadoop_catalog.sales.orders (
        order_id INT,
        customer_id INT,
        amount DOUBLE,
        order_date DATE
    )
    USING iceberg
""")

DataFrame[]

In [29]:
# Visualizando tabelas no schema sales
print("Tabelas no schema 'sales':")
spark.sql("""
    SHOW TABLES IN hadoop_catalog.sales
""").show()

Tabelas no schema 'sales':
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|    sales|   orders|      false|
+---------+---------+-----------+

